# Quick look notebook
---  
*J. Michelle Hu  
University of Utah  
Jan 2025*  

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyproj
import geopandas as gpd
import xarray as xr
import seaborn as sns

from pathlib import PurePath
from s3fs import S3FileSystem, S3Map

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

# Set seaborn palette
sns.set_palette('icefire')

In [ ]:
%reload_ext autoreload
%autoreload 2

### Env setup

In [ ]:
from pathlib import PurePath
# Locate pyproj_datadir for studio env
# From https://stackoverflow.com/questions/69630630/on-fresh-conda-installation-of-pyproj-pyproj-unable-to-set-database-path-pypr
CONDA_ENV = 'studio'
miniconda_dir = '/uufs/chpc.utah.edu/common/home/u6058223/software/pkg/miniconda3'
proj_version = h.fn_list(miniconda_dir, f'envs/{CONDA_ENV}/conda-meta/proj-[0-9]*.json')[0]

VERSION = PurePath(proj_version).stem
pyprojdatadir = f'{miniconda_dir}/pkgs/{VERSION}/share/proj'
print(pyprojdatadir)
pyproj.datadir.set_data_dir(pyprojdatadir)

## Compare Python 3.9 outputs for Yampa and Animas basins

In [ ]:
# Read in yampa time decay depth from csvs
workdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/model_runs'
basin = 'animas'
wy = 2022
basindirs = h.fn_list(workdir, f'{basin}*/wy{wy}/')
csv_fns = [h.fn_list(basindir, '*csv')[0] for basindir in basindirs]
df, df_py39 = [pd.read_csv(csv, parse_dates=True, index_col=0) for csv in csv_fns]

In [ ]:
# loop through each column of the dfs and plot
fig, axa = plt.subplots(int(len(df.columns)), 1, figsize=(12, int(2*len(df.columns))), sharex=True)
for col, ax in zip(df.columns, axa.flatten()):
    # fig, ax = plt.subplots(figsize=(12, 2))
    df[col].plot(ax=ax, label='Default')
    df_py39[col].plot(ax=ax, label='py39', linewidth=0, marker='x', markersize=5)
    ax.set_title(col)
    ax.legend()

In [ ]:
# Now do the same but difference each of them
fig, axa = plt.subplots(int(len(df.columns)), 1, figsize=(12, int(2*len(df.columns))), sharex=True)
for col, ax in zip(df.columns, axa.flatten()):
    # fig, ax = plt.subplots(figsize=(12, 2))
    (df[col] - df_py39[col]).plot(ax=ax, label='Default - py39', color='darkred')
    ax.set_title(col)
    ax.set_ylim(-1e-6, 1e-6)
    ax.legend()

In [ ]:
# Locate the date at which we are seeing the largest differneces
df_diff = df - df_py39
df_diff.idxmax()

In [ ]:
basin = 'yampa'
wy = 2024
basindirs = h.fn_list(workdir, f'{basin}*/wy{wy}/')
csv_fns = [h.fn_list(basindir, '*csv')[0] for basindir in basindirs]
df, df_py39 = [pd.read_csv(csv, parse_dates=True, index_col=0) for csv in csv_fns]

In [ ]:
# loop through each column of the dfs and plot
fig, axa = plt.subplots(int(len(df.columns)), 1, figsize=(12, int(2*len(df.columns))), sharex=True)
for col, ax in zip(df.columns, axa.flatten()):
    # fig, ax = plt.subplots(figsize=(12, 2))
    df[col].plot(ax=ax, label='Default')
    df_py39[col].plot(ax=ax, label='py39', linewidth=0, marker='x', markersize=5)
    ax.set_title(col)
    ax.legend()

In [ ]:
# Now do the same but difference each of them
fig, axa = plt.subplots(int(len(df.columns)), 1, figsize=(12, int(2*len(df.columns))), sharex=True)
for col, ax in zip(df.columns, axa.flatten()):
    # fig, ax = plt.subplots(figsize=(12, 2))
    (df[col] - df_py39[col]).plot(ax=ax, label='Default - py39', color='darkred')
    ax.set_title(col)
    ax.set_ylim(-1e-4, 1e-4)
    ax.legend()

In [ ]:
# Locate the date at which we are seeing the largest differneces
df_diff = df - df_py39
# also print the max difference for that date
print(df_diff.idxmax())
print(df_diff.max())